# Real-World Validation and External Evidence

This analysis evaluates whether the behavioural and portfolio patterns identified in the synthetic Nigerian BNPL dataset are directionally consistent with evidence from established BNPL research and regulatory publications.

The validation does not attempt to reproduce external studies or statistically test equivalence across populations. The Nigerian BNPL dataset is synthetic, while the external studies use real-world populations from other markets with different product definitions, borrower characteristics, regulatory environments and observation periods.

The purpose is therefore directional validation:

**real-world evidence → project hypothesis → observed project finding → directional assessment → limitation**

Primary external sources:

- Consumer Financial Protection Bureau (CFPB), *Consumer Use of Buy Now, Pay Later and Other Unsecured Debt*, 2025
- Laudenbach, Molin, Roszbach and Sondershaus, *Buy Now Pay (Less) Later: Leveraging Private BNPL Data in Consumer Banking*, 2025
- Di Maggio, Katz and Williams, *Buy Now Pay Later Credit: User Characteristics and Effects on Spending Patterns*, 2022

No external study is treated as a benchmark for absolute default rates or model performance.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# =========================================================================
# Real-World Validation
# =========================================================================

# Frozen BNPL analytical outputs
GOLD_PATH = "/Volumes/workspace/default/bnpl_raw/gold_bnpl"
PREDICTIONS_PATH = "/Volumes/workspace/default/bnpl_raw/final_bnpl_test_predictions"
RISK_BANDS_PATH = "/Volumes/workspace/default/bnpl_raw/final_bnpl_risk_bands"
CLUSTERS_PATH = "/Volumes/workspace/default/bnpl_raw/customer_kmeans_clusters"


# =========================================================================
# Load frozen project outputs
# =========================================================================

gold = (
    spark.read
    .format("delta")
    .load(GOLD_PATH)
)

predictions = (
    spark.read
    .format("delta")
    .load(PREDICTIONS_PATH)
)

risk_bands = (
    spark.read
    .format("delta")
    .load(RISK_BANDS_PATH)
)

clusters = (
    spark.read
    .format("delta")
    .load(CLUSTERS_PATH)
)


# =========================================================================
# Reproduce frozen 2024 OOT population
# =========================================================================

validation = (
    gold
    .filter(F.year("purchase_date") == 2024)
    .join(
        predictions.select(
            "transaction_id",
            "default_probability",
            "risk_prediction"
        ),
        on="transaction_id",
        how="inner"
    )
    .join(
        risk_bands.select(
            "transaction_id",
            "risk_decile",
            "risk_band"
        ),
        on="transaction_id",
        how="left"
    )
    .withColumn(
        "default_30d_numeric",
        F.when(
            F.col("default_30d") == True,
            F.lit(1.0)
        ).otherwise(
            F.lit(0.0)
        )
    )
    .withColumn(
        "default_90d_numeric",
        F.when(
            F.col("default_90d") == True,
            F.lit(1.0)
        ).otherwise(
            F.lit(0.0)
        )
    )
)


# =========================================================================
# Input quality audit
# =========================================================================

gold_oot_count = (
    gold
    .filter(F.year("purchase_date") == 2024)
    .count()
)

validation_count = validation.count()

validation_customer_count = (
    validation
    .select("customer_id")
    .distinct()
    .count()
)

validation_null_predictions = (
    validation
    .filter(F.col("default_probability").isNull())
    .count()
)

validation_null_risk_bands = (
    validation
    .filter(F.col("risk_band").isNull())
    .count()
)


print("=" * 75)
print("REAL-WORLD VALIDATION INPUT AUDIT")
print("=" * 75)

print(
    f"2024 Gold transactions:       "
    f"{gold_oot_count:,}"
)

print(
    f"2024 validation transactions: "
    f"{validation_count:,}"
)

print(
    f"Unique validation customers:  "
    f"{validation_customer_count:,}"
)

print(
    f"Missing model probabilities:  "
    f"{validation_null_predictions:,}"
)

print(
    f"Missing risk bands:            "
    f"{validation_null_risk_bands:,}"
)


assert gold_oot_count == 666246
assert validation_count == 666246
assert validation_null_predictions == 0
assert validation_null_risk_bands == 0


# =========================================================================
# Repeat borrowing intensity
# =========================================================================

customer_activity = (
    validation
    .groupBy("customer_id")
    .agg(
        F.count("*").alias(
            "transaction_count"
        ),
        F.sum("principal_ngn").alias(
            "total_exposure_ngn"
        ),
        F.max("default_30d_numeric").alias(
            "any_default_30d"
        ),
        F.max("default_90d_numeric").alias(
            "any_default_90d"
        )
    )
)

repeat_borrowing_summary = (
    customer_activity
    .agg(
        F.count("*").alias(
            "customers"
        ),
        F.avg("transaction_count").alias(
            "avg_transactions_per_customer"
        ),
        F.sum(
            F.when(
                F.col("transaction_count") > 1,
                1
            ).otherwise(0)
        ).alias(
            "repeat_borrowers"
        ),
        F.sum("total_exposure_ngn").alias(
            "total_exposure_ngn"
        )
    )
    .withColumn(
        "repeat_borrower_pct",
        F.col("repeat_borrowers") /
        F.col("customers") *
        100
    )
)


# =========================================================================
# Customer exposure concentration
# =========================================================================

customer_rank_window = Window.orderBy(
    F.col("total_exposure_ngn").desc()
)

exposure_concentration = (
    customer_activity
    .withColumn(
        "exposure_rank",
        F.row_number().over(
            customer_rank_window
        )
    )
)

total_customer_exposure = (
    customer_activity
    .agg(
        F.sum(
            "total_exposure_ngn"
        ).alias(
            "total_exposure"
        )
    )
    .first()["total_exposure"]
)

customer_concentration = (
    exposure_concentration
    .agg(
        F.sum(
            F.when(
                F.col("exposure_rank") <= 10,
                F.col("total_exposure_ngn")
            ).otherwise(0)
        ).alias(
            "top10_exposure"
        ),
        F.sum(
            F.when(
                F.col("exposure_rank") <= 50,
                F.col("total_exposure_ngn")
            ).otherwise(0)
        ).alias(
            "top50_exposure"
        ),
        F.max(
            "total_exposure_ngn"
        ).alias(
            "largest_customer_exposure"
        )
    )
    .withColumn(
        "top10_exposure_pct",
        F.col("top10_exposure") /
        F.lit(total_customer_exposure) *
        100
    )
    .withColumn(
        "top50_exposure_pct",
        F.col("top50_exposure") /
        F.lit(total_customer_exposure) *
        100
    )
    .withColumn(
        "largest_customer_exposure_pct",
        F.col("largest_customer_exposure") /
        F.lit(total_customer_exposure) *
        100
    )
)


# =========================================================================
# Credit-risk information structure
# =========================================================================

risk_information_summary = (
    validation
    .agg(
        F.avg(
            "default_probability"
        ).alias(
            "mean_predicted_pd"
        ),

        F.avg(
            "default_30d_numeric"
        ).alias(
            "observed_default_rate_30d"
        ),

        F.avg(
            "default_90d_numeric"
        ).alias(
            "observed_default_rate_90d"
        ),

        F.avg(
            F.when(
                F.col("first_time_customer") == True,
                F.col("default_30d_numeric")
            )
        ).alias(
            "first_time_default_rate_30d"
        ),

        F.avg(
            F.when(
                F.col("first_time_customer") == False,
                F.col("default_30d_numeric")
            )
        ).alias(
            "repeat_customer_default_rate_30d"
        ),

        F.avg(
            F.when(
                F.col("prior_transaction_count") > 0,
                F.col("default_30d_numeric")
            )
        ).alias(
            "customers_with_prior_history_default_rate_30d"
        ),

        F.avg(
            F.when(
                F.col("prior_transaction_count") == 0,
                F.col("default_30d_numeric")
            )
        ).alias(
            "customers_without_prior_history_default_rate_30d"
        )
    )
)


# =========================================================================
# Behavioural activity and risk
# =========================================================================

behavioural_risk_summary = (
    validation
    .withColumn(
        "activity_group",
        F.when(
            F.col("prior_transaction_count") == 0,
            "No prior BNPL history"
        )
        .when(
            F.col("prior_transaction_count") == 1,
            "1 prior transaction"
        )
        .when(
            F.col("prior_transaction_count") <= 3,
            "2-3 prior transactions"
        )
        .otherwise(
            "4+ prior transactions"
        )
    )
    .groupBy(
        "activity_group"
    )
    .agg(
        F.count("*").alias(
            "transactions"
        ),
        F.countDistinct(
            "customer_id"
        ).alias(
            "customers"
        ),
        F.avg(
            "prior_transaction_count"
        ).alias(
            "avg_prior_transactions"
        ),
        F.avg(
            "prior_total_exposure"
        ).alias(
            "avg_prior_exposure_ngn"
        ),
        F.avg(
            "days_since_previous_transaction"
        ).alias(
            "avg_days_since_previous_transaction"
        ),
        F.avg(
            "default_probability"
        ).alias(
            "avg_predicted_pd"
        ),
        F.avg(
            "default_30d_numeric"
        ).alias(
            "observed_default_rate_30d"
        ),
        F.avg(
            "default_90d_numeric"
        ).alias(
            "observed_default_rate_90d"
        )
    )
    .orderBy(
        "avg_prior_transactions"
    )
)


# =========================================================================
# Risk-band validation
# =========================================================================

risk_band_validation = (
    validation
    .groupBy(
        "risk_band"
    )
    .agg(
        F.count("*").alias(
            "transactions"
        ),
        F.countDistinct(
            "customer_id"
        ).alias(
            "customers"
        ),
        F.sum(
            "principal_ngn"
        ).alias(
            "exposure_ngn"
        ),
        F.avg(
            "default_probability"
        ).alias(
            "avg_predicted_pd"
        ),
        F.avg(
            "default_30d_numeric"
        ).alias(
            "observed_default_rate_30d"
        ),
        F.avg(
            "default_90d_numeric"
        ).alias(
            "observed_default_rate_90d"
        ),
        F.sum(
            F.col("principal_ngn") *
            F.col("default_probability")
        ).alias(
            "pd_weighted_exposure_ngn"
        )
    )
)

risk_band_total_exposure = (
    risk_band_validation
    .agg(
        F.sum(
            "exposure_ngn"
        ).alias(
            "total_exposure"
        ),
        F.sum(
            "pd_weighted_exposure_ngn"
        ).alias(
            "total_pd_weighted_exposure"
        )
    )
    .first()
)

risk_band_validation = (
    risk_band_validation
    .withColumn(
        "exposure_share_pct",
        F.col("exposure_ngn") /
        F.lit(
            risk_band_total_exposure["total_exposure"]
        ) *
        100
    )
    .withColumn(
        "pd_weighted_exposure_share_pct",
        F.col("pd_weighted_exposure_ngn") /
        F.lit(
            risk_band_total_exposure[
                "total_pd_weighted_exposure"
            ]
        ) *
        100
    )
)


# =========================================================================
# Customer segment validation
# =========================================================================

segment_validation = (
    validation
    .join(
        clusters.select(
            "customer_id",
            "cluster"
        ),
        on="customer_id",
        how="left"
    )
    .groupBy(
        "cluster"
    )
    .agg(
        F.count("*").alias(
            "transactions"
        ),
        F.countDistinct(
            "customer_id"
        ).alias(
            "customers"
        ),
        F.sum(
            "principal_ngn"
        ).alias(
            "exposure_ngn"
        ),
        F.avg(
            "default_probability"
        ).alias(
            "avg_predicted_pd"
        ),
        F.avg(
            "default_30d_numeric"
        ).alias(
            "observed_default_rate_30d"
        ),
        F.avg(
            "default_90d_numeric"
        ).alias(
            "observed_default_rate_90d"
        ),
        F.sum(
            F.col("principal_ngn") *
            F.col("default_probability")
        ).alias(
            "pd_weighted_exposure_ngn"
        )
    )
    .withColumn(
        "segment_label",
        F.when(
            F.col("cluster") == 0,
            "Low-Engagement / Low-Exposure"
        )
        .when(
            F.col("cluster") == 1,
            "Active / Higher-Exposure"
        )
        .otherwise(
            "Unclassified"
        )
    )
)

segment_total_exposure = (
    segment_validation
    .agg(
        F.sum(
            "exposure_ngn"
        ).alias(
            "total_exposure"
        ),
        F.sum(
            "pd_weighted_exposure_ngn"
        ).alias(
            "total_pd_weighted_exposure"
        )
    )
    .first()
)

segment_validation = (
    segment_validation
    .withColumn(
        "exposure_share_pct",
        F.col("exposure_ngn") /
        F.lit(
            segment_total_exposure["total_exposure"]
        ) *
        100
    )
    .withColumn(
        "pd_weighted_exposure_share_pct",
        F.col("pd_weighted_exposure_ngn") /
        F.lit(
            segment_total_exposure[
                "total_pd_weighted_exposure"
            ]
        ) *
        100
    )
)


# =========================================================================
# External evidence matrix
# =========================================================================

validation_evidence = spark.createDataFrame(
    [
        (
            "Repeat borrowing",
            "CFPB 2025 reports substantial repeat BNPL use, including 9.5 annual originations per borrower in 2022.",
            "Repeat transaction activity and prior BNPL history are explicitly modelled.",
            "Directly testable",
            "Supports behavioural feature design"
        ),
        (
            "Simultaneous / repeated borrowing",
            "CFPB 2025 reports that approximately 63% of borrowers originated multiple simultaneous BNPL loans at some point in 2022.",
            "The project captures transaction frequency and recent borrowing activity, but does not identify simultaneous outstanding loans.",
            "Directionally relevant",
            "Supports activity and exposure monitoring"
        ),
        (
            "Unsecured debt / exposure",
            "CFPB 2025 finds BNPL users tend to hold higher balances across other unsecured credit categories.",
            "The project explicitly measures prior exposure, recent exposure and portfolio EAD.",
            "Directionally relevant",
            "Supports exposure-based risk analysis"
        ),
        (
            "Credit-risk information in BNPL history",
            "Norges Bank research finds BNPL transaction and repayment information can contribute to internal credit-risk assessment.",
            "The project uses prior transactions, prior exposure, prior defaults and recency variables as risk information.",
            "Strong directional consistency",
            "Supports behavioural risk modelling"
        ),
        (
            "Repayment behaviour",
            "Norges Bank research reports differences in repayment outcomes associated with BNPL experience and delayed BNPL payments.",
            "The project incorporates prior default history and repayment-related behavioural information.",
            "Directional consistency",
            "Supports repayment-history monitoring"
        ),
        (
            "Spending / usage behaviour",
            "Di Maggio, Katz and Williams use transaction-level BNPL data to document usage patterns and effects on spending.",
            "The project focuses on credit risk rather than estimating causal spending effects.",
            "Contextual support only",
            "Useful for business interpretation"
        ),
        (
            "Absolute default rate",
            "CFPB reports approximately 2% average BNPL default for 2019-2022 in its U.S. sample.",
            "The synthetic 2024 OOT portfolio has an observed default rate of approximately 8.0%.",
            "Not directly comparable",
            "Do not benchmark absolute default rates across populations"
        )
    ],
    [
        "evidence_dimension",
        "external_evidence",
        "project_measurement",
        "assessment_type",
        "project_implication"
    ]
)


# =========================================================================
# Display results
# =========================================================================

print("\n" + "=" * 75)
print("PROJECT-SIDE REPEAT BORROWING SUMMARY")
print("=" * 75)

repeat_borrowing_summary.show(
    truncate=False
)


print("\n" + "=" * 75)
print("PROJECT-SIDE CUSTOMER CONCENTRATION")
print("=" * 75)

customer_concentration.show(
    truncate=False
)


print("\n" + "=" * 75)
print("PROJECT-SIDE RISK INFORMATION SUMMARY")
print("=" * 75)

risk_information_summary.show(
    truncate=False
)


print("\n" + "=" * 75)
print("BEHAVIOURAL ACTIVITY AND RISK")
print("=" * 75)

behavioural_risk_summary.show(
    truncate=False
)


print("\n" + "=" * 75)
print("RISK-BAND VALIDATION")
print("=" * 75)

risk_band_validation.orderBy(
    F.when(
        F.col("risk_band") == "Very Low", 1
    )
    .when(
        F.col("risk_band") == "Low", 2
    )
    .when(
        F.col("risk_band") == "Moderate", 3
    )
    .when(
        F.col("risk_band") == "High", 4
    )
    .when(
        F.col("risk_band") == "Very High", 5
    )
    .otherwise(99)
).show(
    truncate=False
)


print("\n" + "=" * 75)
print("SEGMENT VALIDATION")
print("=" * 75)

segment_validation.orderBy(
    "cluster"
).show(
    truncate=False
)


print("\n" + "=" * 75)
print("EXTERNAL EVIDENCE VALIDATION MATRIX")
print("=" * 75)

validation_evidence.show(
    truncate=False
)


# =========================================================================
# Quality gate
# =========================================================================

assert validation_count == 666246
assert validation_customer_count == 421457
assert validation_null_predictions == 0
assert validation_null_risk_bands == 0
assert behavioural_risk_summary.count() == 4
assert risk_band_validation.count() == 5
assert segment_validation.count() == 2
assert validation_evidence.count() == 7


print("\n" + "=" * 75)
print("REAL-WORLD VALIDATION INPUT QUALITY GATE: PASSED")
print("=" * 75)

print("Frozen 2024 OOT population reproduced.")
print("External evidence matrix created.")
print("Behavioural activity evidence generated.")
print("Risk-band validation generated.")
print("Customer-segment validation generated.")
print("No absolute cross-population default-rate benchmarking is used.")

REAL-WORLD VALIDATION INPUT AUDIT
2024 Gold transactions:       666,246
2024 validation transactions: 666,246
Unique validation customers:  421,457
Missing model probabilities:  0
Missing risk bands:            0

PROJECT-SIDE REPEAT BORROWING SUMMARY
+---------+-----------------------------+----------------+---------------------+-------------------+
|customers|avg_transactions_per_customer|repeat_borrowers|total_exposure_ngn   |repeat_borrower_pct|
+---------+-----------------------------+----------------+---------------------+-------------------+
|421457   |1.5808160737631598           |175965          |3.3317913042690205E10|41.751590316449835 |
+---------+-----------------------------+----------------+---------------------+-------------------+


PROJECT-SIDE CUSTOMER CONCENTRATION


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------------+-------------------+-------------------------+-------------------+-------------------+-----------------------------+
|top10_exposure   |top50_exposure     |largest_customer_exposure|top10_exposure_pct |top50_exposure_pct |largest_customer_exposure_pct|
+-----------------+-------------------+-------------------------+-------------------+-------------------+-----------------------------+
|8704875.107734364|3.706427002238047E7|1119428.1055633891       |0.02612671176787408|0.11124427263763806|0.0033598386073254256        |
+-----------------+-------------------+-------------------------+-------------------+-------------------+-----------------------------+


PROJECT-SIDE RISK INFORMATION SUMMARY
+-------------------+-------------------------+-------------------------+---------------------------+--------------------------------+---------------------------------------------+------------------------------------------------+
|mean_predicted_pd  |observed_default_rate_30d|obs

## Real-World Validation Results and Limitations

The real-world validation provides directional support for the project's behavioural and portfolio-risk framework.

Repeat borrowing is material in the synthetic 2024 out-of-time population, with 41.75% of customers classified as repeat borrowers. Customers with greater prior BNPL activity also carry substantially higher historical exposure, increasing from approximately ₦49.9 thousand for customers with one prior transaction to approximately ₦237.5 thousand for customers with four or more prior transactions.

The relationship between prior activity and observed default is comparatively modest. Customers with four or more prior transactions have a 90-day observed default rate of approximately 8.05%, compared with 7.84% for customers without prior BNPL history. This indicates that borrowing frequency in isolation should not be treated as a sufficient risk indicator. Exposure, repayment behaviour, credit information and other borrower characteristics must be considered jointly.

The strongest validation result concerns risk concentration. The Very High risk band represents approximately 25.17% of portfolio exposure but contributes approximately 62.13% of PD-weighted exposure, with a 90-day observed default rate of approximately 38.88%. This supports the use of risk bands and portfolio concentration analysis for targeted risk management.

Customer-level exposure concentration is very low. The ten largest customers account for approximately 0.03% of total exposure, indicating that portfolio risk is driven primarily by the distribution of risk across the portfolio rather than dependence on a small number of individual customers.

External BNPL evidence from the CFPB, Norges Bank and academic research provides directional support for the project's emphasis on repeat borrowing, exposure, repayment history and behavioural information. However, external evidence is not treated as a numerical benchmark because the studies use different countries, populations, product definitions, observation periods and default definitions.

The 8.0% observed default rate in the synthetic Nigerian BNPL portfolio is therefore not interpreted as an estimate of the actual Nigerian BNPL default rate. Similarly, the external studies are not used for model training or calibration.

Overall, the validation supports the economic plausibility of the project's behavioural and portfolio-risk structure while preserving the limitations associated with synthetic data and cross-market comparison.